<a href="https://colab.research.google.com/github/nishta2005/cognicare/blob/main/tabnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, accuracy_score
from imblearn.over_sampling import SMOTE
from pytorch_tabnet.tab_model import TabNetClassifier

df = pd.read_csv('/content/alzheimers_reduced.csv')
df.dropna(inplace=True)
print("Dataset shape:",df.shape)
print("Columns:", df.columns.tolist())





Dataset shape: (2222, 23)
Columns: ['Age', 'Gender', 'Ethnicity', 'EducationLevel', 'BMI', 'PhysicalActivity', 'FamilyHistoryAlzheimers', 'CardiovascularDisease', 'Diabetes', 'Depression', 'HeadInjury', 'SystolicBP', 'MMSE', 'FunctionalAssessment', 'MemoryComplaints', 'BehavioralProblems', 'ADL', 'Confusion', 'Disorientation', 'PersonalityChanges', 'DifficultyCompletingTasks', 'Forgetfulness', 'Diagnosis']


In [ ]:
!pip install pytorch-tabnet --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.6 MB/s eta 0:00:00


In [ ]:
le = LabelEncoder()
df['Diagnosis'] = le.fit_transform(df['Diagnosis'])

X = df.drop('Diagnosis', axis=1).values.astype(np.float32)
y = df['Diagnosis'].values

sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X, y)

print("Classes found:", le.classes_)
print("Balanced class counts:", np.bincount(y_res))

Classes found: [0 1]
Balanced class counts: [1111 1111]


In [8]:
start = time.time()

tabnet_model = TabNetClassifier(
    n_d=16,           # width of decision step — increase to 32 if accuracy is low
    n_a=16,           # width of attention step — keep same as n_d
    n_steps=3,        # number of sequential attention steps
    gamma=1.3,        # coefficient for feature reusage across steps
    seed=42,
    verbose=1         # shows training progress each epoch
)

tabnet_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric=['accuracy'],
    max_epochs=100,
    patience=20,      # stops early if no improvement for 20 epochs
    batch_size=64,
    virtual_batch_size=32
)

tabnet_time = time.time() - start
print(f"\nTraining time: {tabnet_time:.2f}s")

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.88891 | val_0_accuracy: 0.5573  |  0:00:01s
epoch 1  | loss: 0.62022 | val_0_accuracy: 0.49438 |  0:00:01s
epoch 2  | loss: 0.59337 | val_0_accuracy: 0.6764  |  0:00:02s
epoch 3  | loss: 0.55713 | val_0_accuracy: 0.67865 |  0:00:03s
epoch 4  | loss: 0.55806 | val_0_accuracy: 0.70562 |  0:00:03s
epoch 5  | loss: 0.50147 | val_0_accuracy: 0.75955 |  0:00:04s
epoch 6  | loss: 0.48749 | val_0_accuracy: 0.80674 |  0:00:05s
epoch 7  | loss: 0.475   | val_0_accuracy: 0.8382  |  0:00:06s
epoch 8  | loss: 0.45666 | val_0_accuracy: 0.85393 |  0:00:06s
epoch 9  | loss: 0.435   | val_0_accuracy: 0.8382  |  0:00:07s
epoch 10 | loss: 0.43383 | val_0_accuracy: 0.83146 |  0:00:08s
epoch 11 | loss: 0.40809 | val_0_accuracy: 0.86742 |  0:00:08s
epoch 12 | loss: 0.39396 | val_0_accuracy: 0.84719 |  0:00:09s
epoch 13 | loss: 0.38675 | val_0_accuracy: 0.84944 |  0:00:10s
epoch 14 | loss: 0.3779  | val_0_accuracy: 0.85618 |  0:00:10s
epoch 15 | loss: 0.36479 | val_0_accuracy: 0.86966 |  0

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Training time: 58.40s


In [11]:
feature_importances = tabnet_model.feature_importances_
feature_names = df.drop('Diagnosis', axis=1).columns.tolist()

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values('Importance', ascending=False)

print("\nTop features TabNet paid attention to:")
print(importance_df.to_string(index=False))


Top features TabNet paid attention to:
                  Feature   Importance
         MemoryComplaints 2.755178e-01
                     MMSE 1.997964e-01
                      ADL 1.848993e-01
     FunctionalAssessment 1.283281e-01
                Ethnicity 5.810116e-02
               Depression 5.103557e-02
    CardiovascularDisease 3.333398e-02
               SystolicBP 2.519383e-02
           EducationLevel 1.608519e-02
                Confusion 1.258416e-02
                      Age 8.264167e-03
                      BMI 3.249054e-03
                   Gender 2.513721e-03
         PhysicalActivity 7.782770e-04
            Forgetfulness 2.708248e-04
                 Diabetes 2.859577e-05
       PersonalityChanges 1.897353e-05
DifficultyCompletingTasks 9.744896e-07
               HeadInjury 0.000000e+00
  FamilyHistoryAlzheimers 0.000000e+00
       BehavioralProblems 0.000000e+00
           Disorientation 0.000000e+00


In [14]:
# TabNet has its own save method
tabnet_model.save_model('tabnet_model')

# save label encoder separately
import joblib
joblib.dump(le, 'label_encoder_tabnet.pkl')

from google.colab import files
files.download('tabnet_model.zip')
files.download('label_encoder_tabnet.pkl')

Successfully saved model at tabnet_model.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>